# Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Lưu lượng Internet và cảnh báo sự cố 20 quốc gia (Cloudflare Radar)

**Khung SDG**: SDG 9: hạ tầng số tin cậy; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45) theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [ ]:
import os
os.chdir(r'D:\Semester3\ADY201m\Project')  

# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

In [ ]:
# Step 1b (SỬA): Cloudflare Radar API - hourly HTTP traffic for 20 countries, 3 NĂM gần nhất (09/2023-09/2026)
from dotenv import load_dotenv
import os, time
import pandas as pd, requests
load_dotenv()
H = {'Authorization': f'Bearer {os.environ["CF_API_TOKEN"]}'}
LOCS = ['US','CA','MX','BR','GB','DE','FR','NL','ES','IT','IN','JP','KR','SG','AU','VN','ID','ZA','NG','AE']


end = pd.Timestamp.utcnow().floor('h')
start = end - pd.DateOffset(years=3)
chunk = pd.Timedelta(weeks=4)
windows = []
w0 = start
while w0 < end:
    w1 = min(w0 + chunk, end)
    windows.append((w0, w1))
    w0 = w1
print(f'{len(windows)} windows x {len(LOCS)} countries = {len(windows)*len(LOCS)} API calls')
print('Từ:', start.date(), 'đến:', end.date())

def fetch_window(loc, w0, w1, max_retries=3):
    params = {
        'aggInterval': '1h',
        'location': loc,
        'format': 'json',
        'dateStart': w0.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'dateEnd': w1.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }
    for attempt in range(max_retries):
        r = requests.get('https://api.cloudflare.com/client/v4/radar/netflows/timeseries',
                          params=params, headers=H, timeout=120).json()
        if r.get('success'):
            return r
        errors = r.get('errors', [])
        is_rate_limit = any(e.get('code') == 1015 for e in errors)
        if is_rate_limit and attempt < max_retries - 1:
            wait = 5 * (attempt + 1)
            print(f'  rate limited on {loc} {w0.date()}-{w1.date()}, retrying in {wait}s...')
            time.sleep(wait)
            continue
        print('FAILED', loc, w0.date(), w1.date(), errors)
        return None
    return None

frames = []
for loc in LOCS:
    for w0, w1 in windows:
        r = fetch_window(loc, w0, w1)
        if r is None:
            continue
        s = r['result']['serie_0']
        d = pd.DataFrame({'ts': pd.to_datetime(s['timestamps']), 'traffic': [float(v) for v in s['values']]})
        d['loc'] = loc
        frames.append(d)
        time.sleep(0.3)
    print(loc, 'done')

df = pd.concat(frames, ignore_index=True)
print('traffic table:', df.shape)

# Outage annotations: chia làm 3 lần gọi, mỗi lần 1 năm (dateRange rút gọn tối đa chỉ tới 52w)
ann_frames = []
y0 = start
while y0 < end:
    y1 = min(y0 + pd.DateOffset(years=1), end)
    ann = requests.get('https://api.cloudflare.com/client/v4/radar/annotations/outages',
                        params={'dateStart': y0.strftime('%Y-%m-%dT%H:%M:%SZ'),
                                'dateEnd': y1.strftime('%Y-%m-%dT%H:%M:%SZ'),
                                'limit': 1000, 'format': 'json'},
                        headers=H, timeout=120).json()
    if ann.get('success'):
        ann_frames.append(pd.DataFrame(ann['result']['annotations']))
    else:
        print('annotations FAILED', y0.date(), y1.date(), ann.get('errors'))
    y0 = y1
    time.sleep(0.3)
out = pd.concat(ann_frames, ignore_index=True) if ann_frames else pd.DataFrame()
print(df.shape, out.shape)


In [4]:
df = pd.read_parquet('data/processed/radar.parquet')
print(df.shape)

(526060, 5)


### 1.1b. Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan trước khi tự viết EDA ở các ô sau.

In [5]:
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 238.32it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

In [ ]:
# Step 2: hourly table per location, outage flags from annotations, major cloud incident flags (compiled from AWS, Azure, Google Cloud status history), about 175 thousand location-hours per year
df = df.drop_duplicates(['loc','ts']).sort_values(['loc','ts']); df['ts'] = df.ts.dt.tz_localize(None)
out['start'] = pd.to_datetime(out.startDate).dt.tz_localize(None); out['end'] = pd.to_datetime(out.endDate).dt.tz_localize(None)
df['outage'] = 0
for r in out.itertuples():
    locs = [l.get('code') for l in (r.locationsDetails if isinstance(r.locationsDetails, list) else [])]
    df.loc[df['loc'].isin(locs) & (df.ts >= r.start) & (df.ts <= r.end), 'outage'] = 1
inc = pd.read_csv('data/raw/cloud_incidents.csv', parse_dates=['start','end'])       # provider, region, start, end, severity (compiled from public status pages)
df['cloud_inc'] = 0
for r in inc.itertuples(): df.loc[(df.ts >= r.start) & (df.ts <= r.end), 'cloud_inc'] = 1
df.to_parquet('data/processed/radar.parquet', index=False); print(df.shape, 'outage share', df.outage.mean().round(4))

### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [6]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
_, cleaning_log = clean_log(df, [
    ('drop duplicates', lambda x: x.drop_duplicates(['loc', 'ts']), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['traffic']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates       526060      526060        0  same unit and timestamp
1  drop missing target       526060      526060        0       cannot be forecast


### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [7]:
eda = df.copy()
import io
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T                         # numeric statistics
desc_num['skew'] = eda.select_dtypes('number').skew()  # skewness, > 1 means strongly right-skewed
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude='number').describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in eda.select_dtypes('number').columns:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False); print(notes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 526060 entries, 0 to 526059
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         526060 non-null  datetime64[ns]
 1   traffic    526060 non-null  float64       
 2   loc        526060 non-null  object        
 3   outage     526060 non-null  int64         
 4   cloud_inc  526060 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 20.1+ MB

              count                 mean                  min                  25%                  50%                  75%                  max       std  \
ts           526060  2025-03-16 07:00:00  2023-09-15 08:00:00  2024-06-15 07:00:00  2025-03-16 07:00:00  2025-12-15 07:00:00  2026-09-15 06:00:00       NaN   
traffic    526060.0             0.531068                  0.0             0.360147             0.541041             0.704383                  1.0   0.214

### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [8]:
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv('report/table_missing.csv'); print(miss.head(15))
# missingness of the target by time and by unit
tcol, gcol = 'ts', 'loc'
by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['traffic'].apply(lambda s: s.isna().mean())
by_unit = eda.groupby(gcol)['traffic'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())
print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())
# co-missingness matrix: do columns go missing together
co = eda.isna().astype(int); co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1: print(co.corr().round(2))
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D'); ax.set_ylabel('Missing rate'); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig('report/fig_eda_missing.png', dpi=300)

           missing_rate decision
ts                  0.0     keep
traffic             0.0     keep
loc                 0.0     keep
outage              0.0     keep
cloud_inc           0.0     keep
target missing by period (top): {Period('2023-09', 'M'): 0.0, Period('2023-10', 'M'): 0.0, Period('2023-11', 'M'): 0.0, Period('2023-12', 'M'): 0.0, Period('2024-01', 'M'): 0.0}
target missing by unit (top): {'AE': 0.0, 'AU': 0.0, 'BR': 0.0, 'CA': 0.0, 'DE': 0.0}


### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [9]:
num_cols = [c for c in ['traffic', 'outage', 'cloud_inc'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'traffic': cols.append([c, 'target', 'keep']); continue
    if c in ('ts', 'loc'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'traffic'] if 'traffic' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('loc')['traffic'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['ts'] if np.issubdtype(eda['ts'].dtype, np.number) else pd.to_datetime(eda['ts'], errors='coerce').dt.month
by_t = eda.groupby(period)['traffic'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

           traffic  outage  cloud_inc
traffic      1.000   0.005     -0.001
outage       0.005   1.000     -0.006
cloud_inc   -0.001  -0.006      1.000
      mean  median    p05    p95        n
loc                                      
NL   0.662   0.689  0.410  0.889  26303.0
SG   0.648   0.687  0.295  0.901  26303.0
US   0.638   0.645  0.395  0.869  26303.0
ID   0.636   0.666  0.321  0.889  26303.0
DE   0.610   0.642  0.303  0.883  26303.0
IN   0.595   0.674  0.182  0.889  26303.0
KR   0.589   0.611  0.282  0.872  26303.0
VN   0.577   0.617  0.221  0.883  26303.0
FR   0.552   0.572  0.238  0.857  26303.0
JP   0.551   0.545  0.290  0.854  26303.0
     mean  median    p05    p95        n
ts                                      
1   0.543   0.555  0.182  0.877  44640.0
2   0.526   0.528  0.173  0.870  40800.0
3   0.538   0.552  0.184  0.860  44640.0
4   0.540   0.552  0.186  0.867  43200.0
5   0.509   0.515  0.167  0.837  44640.0
6   0.512   0.519  0.171  0.839  43200.0
7   0.551   0.57

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [10]:
# SQL statements run one by one in DuckDB; each SELECT prints its first 20 rows and is saved to report/table_{label}.csv
import duckdb
if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')   # reuse the connection opened in Step 1
SQL = r"""
-- Step 3: sql/queries.sql
CREATE OR REPLACE TABLE rd AS SELECT * FROM 'data/processed/radar.parquet';
-- Q1a: weekly traffic profile per location
SELECT loc, dayofweek(ts) AS dow, hour(ts) AS hr, AVG(traffic) AS traffic FROM rd GROUP BY 1, 2, 3 ORDER BY 1, 2, 3;
-- Q1b: traffic drop during outage hours
SELECT outage, AVG(traffic / NULLIF(w_mean, 0)) AS rel_traffic FROM (SELECT *, AVG(traffic) OVER (PARTITION BY loc, dayofweek(ts), hour(ts)) AS w_mean FROM rd) GROUP BY 1;
-- Q2: cloud incident hours: relative traffic change
SELECT cloud_inc, AVG(traffic / NULLIF(w_mean, 0)) AS rel_traffic FROM (SELECT *, AVG(traffic) OVER (PARTITION BY loc, dayofweek(ts), hour(ts)) AS w_mean FROM rd) GROUP BY 1;
-- Q3: features and targets: traffic 1 h and 24 h ahead, anomaly (relative drop below 0.7) in the next 3 h
CREATE OR REPLACE TABLE feat AS
SELECT loc, ts, traffic, outage, cloud_inc, hour(ts) AS hr, dayofweek(ts) AS dow,
       LAG(traffic, 1) OVER w AS t_lag1, LAG(traffic, 24) OVER w AS t_lag24, LAG(traffic, 168) OVER w AS t_lag168, AVG(traffic) OVER (w ROWS BETWEEN 23 PRECEDING AND CURRENT ROW) AS t_ma24,
       traffic / NULLIF(AVG(traffic) OVER (w ROWS BETWEEN 167 PRECEDING AND CURRENT ROW), 0) AS rel_w, LEAD(traffic, 1) OVER w AS y_1h, LEAD(traffic, 24) OVER w AS y_24h,
       (MIN(traffic) OVER (w ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING) / NULLIF(AVG(traffic) OVER (w ROWS BETWEEN 167 PRECEDING AND CURRENT ROW), 0) < 0.7)::INT AS y_drop_3h
FROM rd WINDOW w AS (PARTITION BY loc ORDER BY ts);
-- Q4: row counts and share of anomaly label
SELECT COUNT(*) AS n, COUNT(y_24h) AS n24, AVG(y_drop_3h) AS drop_share FROM feat;
"""
labels = ['Q1a_weekly_profile', 'Q1b_outage_rel_traffic', 'Q2_cloud_inc_rel_traffic', 'Q3_drop_share_summary']
SQL_NO_COMMENTS = '\n'.join(l for l in SQL.splitlines() if not l.strip().startswith('--'))   # drop comment lines first (they may contain semicolons)
statements = [s.strip() for s in SQL_NO_COMMENTS.split(';') if s.strip()]
k = 0
for stmt in statements:
    res = con.execute(stmt)
    if stmt.upper().startswith(('SELECT', 'WITH')):
        label = labels[k] if k < len(labels) else f'Q{k+1}'
        out = res.df()
        out.to_csv(f'report/table_{label}.csv', index=False)
        print(f'--- {label} ---')
        print(out.head(20))
        k += 1
open('sql/queries.sql', 'w').write(SQL)


--- Q1a_weekly_profile ---
   loc  dow  hr   traffic
0   AE    0   0  0.270834
1   AE    0   1  0.239870
2   AE    0   2  0.233483
3   AE    0   3  0.251061
4   AE    0   4  0.284634
5   AE    0   5  0.334863
6   AE    0   6  0.393509
7   AE    0   7  0.456881
8   AE    0   8  0.504795
9   AE    0   9  0.561097
10  AE    0  10  0.613177
11  AE    0  11  0.621936
12  AE    0  12  0.599086
13  AE    0  13  0.606429
14  AE    0  14  0.621124
15  AE    0  15  0.648786
16  AE    0  16  0.689217
17  AE    0  17  0.712200
18  AE    0  18  0.694594
19  AE    0  19  0.656345
--- Q1b_outage_rel_traffic ---
   outage  rel_traffic
0       0     0.999956
1       1     1.009062
--- Q2_cloud_inc_rel_traffic ---
   cloud_inc  rel_traffic
0          0     1.000090
1          1     0.991213
--- Q3_drop_share_summary ---
        n     n24  drop_share
0  526060  525580    0.306686


1642

### 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định

**Làm gì**: từ kết quả Q1, Q2 dựng bảng có số thứ tự đưa thẳng vào bài; kiểm định khác biệt giữa nhóm (Kruskal-Wallis, không cần giả định chuẩn) và tương quan Spearman giữa mục tiêu và các biến ngoại sinh kèm p-value.

**Đọc thế nào**: p < 0,05 nghĩa là khác biệt giữa nhóm khó do ngẫu nhiên; hệ số Spearman cho chiều và độ mạnh quan hệ đơn điệu; kết luận RQ1 phải là một câu có số, ví dụ "mùa đông cao gấp 1,9 lần mùa hè, p < 0,001".

In [11]:
from scipy.stats import kruskal, spearmanr
q1 = con.execute("SELECT * FROM feat").df()          # feature table created in Q3
if np.issubdtype(q1['ts'].dtype, np.number): q1['period'] = q1['ts']            # numeric time key (year, cycle)
else: q1['period'] = pd.to_datetime(q1['ts'], errors='coerce').dt.month             # datetime key: use month as period
# Table RQ1a: target by period
rq1a = q1.groupby('period')['traffic'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.values for _, s in q1.groupby('period')['traffic'] if len(s) > 30]
if len(groups) > 1: print('Kruskal-Wallis across periods:', kruskal(*groups))
# Table RQ1b: Spearman correlation of the target with exogenous variables
# FIX: nan_policy='omit' của spearmanr rất chậm trên mảng lớn có NaN (known scipy performance issue) -> tự dropna trước
ext = [c for c in q1.select_dtypes('number').columns if c not in ('traffic', 'y_24h') and not c.startswith('traffic')]
rows = []
for c in ext[:15]:
    sub = q1[[c, 'traffic']].dropna()
    r, pv = spearmanr(sub[c], sub['traffic'])
    rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print(rq1b)
# Table RQ1c: target by unit, top and bottom 5
rq1c = q1.groupby('loc')['traffic'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('RQ1 CONCLUSION (fill with real numbers): highest period', rq1a['mean'].idxmax(), 'is', round(rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9), 2), 'times the lowest period')


         mean  median    std  count
period                             
1       0.543   0.555  0.221  44640
2       0.526   0.528  0.221  40800
3       0.538   0.552  0.213  44640
4       0.540   0.552  0.215  43200
5       0.509   0.515  0.210  44640
6       0.512   0.519  0.209  43200
7       0.551   0.570  0.208  44640
8       0.572   0.593  0.207  44640
9       0.549   0.560  0.206  43180
10      0.521   0.528  0.207  44640
11      0.477   0.468  0.221  43200
12      0.533   0.540  0.216  44640
Kruskal-Wallis across periods: KruskalResult(statistic=np.float64(5863.241303293489), pvalue=np.float64(0.0))
     variable  spearman_r       p_value
9        y_1h       0.948  0.000000e+00
4      t_lag1       0.948  0.000000e+00
5     t_lag24       0.944  0.000000e+00
6    t_lag168       0.917  0.000000e+00
8       rel_w       0.810  0.000000e+00
7      t_ma24       0.552  0.000000e+00
10  y_drop_3h      -0.498  0.000000e+00
2          hr       0.158  0.000000e+00
11     period      -0.022 

## Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

Mỗi hình gắn với một ý của RQ1 và có caption là kết luận. Bộ hình: (1) phân phối mục tiêu, histogram và boxplot; (2) chuỗi theo thời gian của vài đơn vị; (3) heatmap thời kỳ x đơn vị; (4) boxplot mục tiêu theo thời kỳ; (5) scatter mục tiêu với biến ngoại sinh mạnh nhất; (6) heatmap tương quan; (7) so sánh nhóm bằng cột có số. Quy tắc: chữ tiếng Anh, bỏ viền trên và phải, số in đậm trên cột, 300 dpi, tên file `fig_rq1_*.png`.

In [12]:
import seaborn as sns
def style(ax): ax.spines[['top', 'right']].set_visible(False)
d = q1.dropna(subset=['traffic'])
# (1) distribution
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['traffic'], bins=50, color='#1B6B6D'); ax[0].set_xlabel('traffic '); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['traffic'], vert=False); ax[1].set_xlabel('traffic'); style(ax[1])
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)
# (2) time series of the first 3 units
units = d['loc'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['loc'] == u].sort_values('ts'); ax.plot(s['ts'], s['traffic'], lw=0.8, label=str(u))
ax.set_ylabel('traffic'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)
# (3) heatmap period x unit (top 15 units)
top = d.groupby('loc')['traffic'].mean().nlargest(15).index
hm = d[d['loc'].isin(top)].pivot_table(index='loc', columns='period', values='traffic', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4)); sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'traffic'}); ax.set_xlabel('Period'); ax.set_ylabel('Unit')
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)
# (4) boxplot by period
fig, ax = plt.subplots(figsize=(9, 3)); sns.boxplot(data=d, x='period', y='traffic', color='#9FBFBF', ax=ax, showfliers=False); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)
# (5) scatter against the strongest exogenous variable
if len(rq1b):
    v = rq1b.iloc[0].variable
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.scatter(d[v], d['traffic'], s=4, alpha=0.3, color='#1B6B6D'); ax.set_xlabel(v); ax.set_ylabel('traffic'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)
# (6) correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(d.select_dtypes('number').corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)
# (7) group comparison with labelled bars
gm = d.groupby('period')['traffic'].mean()
fig, ax = plt.subplots(figsize=(8, 3)); bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D'); ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8); ax.set_ylabel('Mean traffic'); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)
print('7 figures saved to report/. Write each caption as: conclusion + number + condition.')

7 figures saved to report/. Write each caption as: conclusion + number + condition.


## Test case của Notebook_A (chạy trước khi bàn giao)

Mỗi assert là một điều kiện phải đúng; nếu sai, sửa bước tương ứng rồi chạy lại. Sau khi qua hết, ô cuối tạo `manifest.json` để Sinh viên B kiểm tra.

In [13]:
import hashlib, json, os
feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['y_24h'].notna().sum() > 0.8 * len(feat), 'too many missing targets'
key = ['loc', 'ts']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'
lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'y_24h']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {'rows': int(len(feat)), 'cols': list(feat.columns), 'md5': h,
            'time_min': str(feat['ts'].min()), 'time_max': str(feat['ts'].max()), 'author': 'Student A'}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str); print(manifest)

Tests A: OK
{'rows': 526060, 'cols': ['loc', 'ts', 'traffic', 'outage', 'cloud_inc', 'hr', 'dow', 't_lag1', 't_lag24', 't_lag168', 't_ma24', 'rel_w', 'y_1h', 'y_24h', 'y_drop_3h'], 'md5': '46b339f482985101208a78e4877746d0', 'time_min': '2023-09-15 08:00:00', 'time_max': '2026-09-15 06:00:00', 'author': 'Student A'}


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [ ] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [ ] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [ ] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [ ] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [ ] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [ ] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [ ] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [ ] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [ ] Audit Log cá nhân có đủ prompt của tuần
- [ ] Báo cáo năm mục đã dán vào kênh nhóm trước 12:00 thứ Tư hoặc thứ Bảy


## Mẫu báo cáo tiến độ (slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45), 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................